In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/dias-rapids-25.02/lib/python3.10/site-packages/IPython/core/extensions.py:145: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  mod.load_ipython_extension(self.shell)


In [3]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-taxi/annotated/checkpoints/post_cell_17.pickle

trying: ['map_payment_type']
me:  17
trying: ['pd']
me:  0
trying: ['file_loc']
me:  1
trying: ['trip_data']


me:  21
trying: ['plt']
me:  0
trying: ['continuous_columns']
me:  23
trying: ['time']
me:  0
trying: ['orig_output']
me:  36
trying: ['sns']
me:  0


Declaring variable pd
Declaring variable plt
Declaring variable time
Declaring variable sns
Declaring variable file_loc
Declaring variable map_payment_type
Declaring variable trip_data
Declaring variable continuous_columns
Declaring variable orig_output


In [4]:
%%RecordEvent
%%time
### cell 18 ###

print(trip_data.shape)
# Filter out negative fares with direct boolean indexing
trip_data = trip_data[trip_data.fare_amount >= 0]
trip_data.shape, trip_data.head()

(8759874, 16)
CPU times: user 24.5 ms, sys: 24.2 ms, total: 48.7 ms
Wall time: 55.3 ms


((8755614, 16),
   tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  trip_distance  \
 0  2018-01-01 00:21:05   2018-01-01 00:24:23                1            0.5   
 1  2018-01-01 00:44:55   2018-01-01 01:03:05                1            2.7   
 2  2018-01-01 00:08:26   2018-01-01 00:14:21                2            0.8   
 3  2018-01-01 00:20:22   2018-01-01 00:52:51                1           10.2   
 4  2018-01-01 00:09:18   2018-01-01 00:27:06                2            2.5   
 
    PULocationID  DOLocationID payment_type  fare_amount  tip_amount  \
 0            41            24         Cash          4.5        0.00   
 1           239           140         Cash         14.0        0.00   
 2           262           141  Credit_card          6.0        1.00   
 3           140           257         Cash         33.5        0.00   
 4           246           239  Credit_card         12.5        2.75   
 
    tolls_amount  total_amount   duration  trip_pickup_hour  t

In [5]:
%Checkpoint /home/dias-benchmarks/new_notebooks/nyc-taxi/rewritten/o4_mini_high/checkpoints/post_cell_18_try_0.pickle

migration speed (bps): 792937026.2010602
---------------------------
variables to migrate:
time 72
file_loc 113
continuous_columns 120
pd 72
orig_output 48
map_payment_type 144
plt 72
trip_data 48
sns 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/new_notebooks/nyc-taxi/rewritten/o4_mini_high/checkpoints/post_cell_18_try_0.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['trip_data']
Intermediate variables ['file_loc']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['trip_data']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['trip_data']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables ['trip_data']
Intermediate variables []
Future variables []
Modified dataframes
  trip_data
    Input columns: set()
    Changed columns: set()
    Created columns: set()
    Deleted columns: {'VendorID', 'RatecodeID', 'store_and_fwd_flag'}
======= Cell 4 =======
Input variables []
Active variables ['trip_data']
Intermediate variables []
Future variables []
Modified dataframes
  trip_data
    Input columns: set()
    Changed columns: {'tpep_dropoff_datetime', 'tpep_pickup_datetime'}
    Created columns: set()

In [7]:
with open(
    "/home/dias-benchmarks/new_notebooks/nyc-taxi/opt_cell_exec_info_18_try_0.pkl", "wb"
) as f:
    pickle.dump(opt_cell_exec_info[18], f)

In [ ]:
opt_output = Out.get(4)

In [ ]:
trip_data_opt = trip_data
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-taxi/annotated/checkpoints/post_cell_18.pickle
assert compare_df(trip_data_opt, trip_data)

import numpy as np
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output